# Norway/Base/Smiley Four-Panel Plot\n\nLoads saved tensors and draws the requested four panels. Tweak the plotting knobs in the second cell.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from data import COLOR_NAMES, NUM_CLASSES
from reward import norwegian_region_scores
from reward_smiley import smiley_scores

DATA_PATH = Path("figures/norway_base_smiley_four_panel_data.pt")
OUT_PATH = Path("figures/norway_base_smiley_four_panel_tweak.png")

bundle = torch.load(DATA_PATH, map_location="cpu")
panels = bundle["panels"]
print(bundle["description"])
print({k: tuple(v.shape) for k, v in panels.items()})
print(bundle["sources"])

Four-panel plotting data: Gaussian prior, uniform base, Norway guided, smiley guided.
{'prior': (10000, 5), 'base': (10000, 5), 'norway': (10000, 5), 'smiley': (1000, 5)}
{'norway': 'figures/uniform_norway_slow_color_guidance_N10000_freeze_t08_ratio2_endpoints.pt', 'smiley': 'figures/uniform_smiley_guidance_N1000_lam0p2_beta16_mc16_steps200_glass8_chunk100_endpoints.pt', 'prior': 'torch.randn with seed=0, shape matching Norway endpoints'}


In [12]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.serif": ["Computer Modern Roman", "DejaVu Serif"],
})

In [ ]:
# Plot knobs.
PLOT_COLORS = {
    "blue": "#006DCC",
    "white": "#F8F8F8",
    "red": "#FF2D20",
}
TITLES = [
      r"$p_0(x)$",                                                                                                                               
      r"$p_1(x)$",
  r"$p_1(x)\,e^{r_{\mathrm{Norway}}(x)}$",
  r"$p_1(x)\,e^{r_{\mathrm{Smiley}}(x)}$",
  ]
PANEL_KEYS = ["prior", "base", "norway", "smiley"]
PANEL_KINDS = ["prior", "base", "norway", "smiley"]
LIMITS = {
    "prior": (-3.0, 3.0),
    "base": (-1.08, 1.08),
    "norway": (-1.08, 1.08),
    "smiley": (-1.08, 1.08),
}
MARKER_SIZE = {
    "prior": 3.0,
    "base": 3.0,
    "norway": 3.0,
    "smiley": 3.0,
}
ALPHA = 0.90
FIGSIZE = (10.3, 3.35)                                                                                                                         
WSPACE = 0.0    
TITLE_SIZE=25
LABEL_FONTSIZE=25
DPI = 220



def panel_target(x, kind):
    y = x[:, NUM_CLASSES:]
    if kind == "norway":
        scores = norwegian_region_scores(y[:, 0], y[:, 1], order="model")
        return scores.argmax(dim=-1), scores.max(dim=-1).values
    if kind == "smiley":
        scores = smiley_scores(y[:, 0], y[:, 1])
        return scores.argmax(dim=-1), scores.max(dim=-1).values
    return None, torch.zeros(x.shape[0])


def plot_panel(ax, x, title, kind):
      c = x[:, :NUM_CLASSES]
      y = x[:, NUM_CLASSES:]
      pred = c.argmax(dim=-1)

      if kind == "smiley":
          scores = smiley_scores(y[:, 0], y[:, 1])  # blue, white, red
          target = scores.argmax(dim=-1)
          correct = pred == target

          blue_right_eye = correct & (target == 0)
          white_mouth = correct & (target == 1)
          red_left_eye = correct & (target == 2)

          # Base smiley points only, faint.
          for color_idx, name in enumerate(COLOR_NAMES):
              mask = pred == color_idx
              ax.scatter(
                  y[mask, 0], y[mask, 1],
                  s=MARKER_SIZE["smiley"],
                  c=PLOT_COLORS[name],
                  alpha=0.35 if name == "red" else 0.55,
                  edgecolors="#AFAFAF" if name == "white" else "none",
                  linewidths=0.08 if name == "white" else 0.0,
                  rasterized=True,
                  zorder=1,
              )

          ax.scatter(
              y[red_left_eye, 0], y[red_left_eye, 1],
              s=MARKER_SIZE["smiley"],
              c="#C94A42",
              alpha=0.65,
              edgecolors="none",
              rasterized=True,
              zorder=3,
          )

          ax.scatter(
              y[blue_right_eye, 0], y[blue_right_eye, 1],
              s=MARKER_SIZE["smiley"] * 1.15,
              c=PLOT_COLORS["blue"],
              alpha=0.95,
              edgecolors="none",
              rasterized=True,
              zorder=5,
          )

          ax.scatter(
              y[white_mouth, 0], y[white_mouth, 1],
              s=MARKER_SIZE["smiley"] * 1.15,
              c=PLOT_COLORS["white"],
              alpha=0.95,
              edgecolors="#AFAFAF",
              linewidths=0.08,
              rasterized=True,
              zorder=6,
          )

      else:
          # Other plots stay exactly color-by-color, no accuracy/layering logic.
          y_np = y.numpy()
          pred_np = pred.numpy()
          for color_idx, name in enumerate(COLOR_NAMES):
              mask = pred_np == color_idx
              ax.scatter(
                  y_np[mask, 0], y_np[mask, 1],
                  s=MARKER_SIZE[kind],
                  c=PLOT_COLORS[name],
                  alpha=ALPHA,
                  edgecolors="#AFAFAF" if name == "white" else "none",
                  linewidths=0.03 if name == "white" else 0.0,
                  rasterized=True,
              )

      ax.set_facecolor("#050505")
      ax.set_xlim(*LIMITS[kind])
      ax.set_ylim(*LIMITS[kind])
      ax.set_aspect("equal")
      ax.set_title(title, fontsize=TITLE_SIZE, pad=5)
      ax.set_xlabel(r"$y_1$", fontsize=LABEL_FONTSIZE, labelpad=0)
      ax.set_ylabel(r"$y_2$", fontsize=LABEL_FONTSIZE, labelpad=0)
      ax.set_xticks([])
      ax.set_yticks([])



fig, axes = plt.subplots(1, 4, figsize=FIGSIZE, constrained_layout=False)                                                                      
for i, (ax, key, title, kind) in enumerate(zip(axes, PANEL_KEYS, TITLES, PANEL_KINDS)):
    plot_panel(ax, panels[key], title, kind)                                                                                                   
    if i != 0:                                                                                                                                                                                                                                                       
        ax.set_ylabel("")                                                                                                                      
                
fig.subplots_adjust(left=0.001, right=0.999, bottom=0.13, top=0.88, wspace=0.0)                                                                
fig.savefig(OUT_PATH, dpi=DPI)
OUT_PATH

PosixPath('figures/norway_base_smiley_four_panel_tweak.png')